In [ ]:
import numpy as np # 파일 상단에 이 임포트가 있는지 확인
from PIL import Image, UnidentifiedImageError # UnidentifiedImageError 추가
from tqdm import tqdm
from pathlib import Path
import torch
import torchvision.transforms as T
import json

# (get_organ_detector_model 함수는 기존과 동일하게 필요합니다)
# ...

# =================================================================
# PART 2 & 3: 카메라 이동 예측 및 JSON 저장 (마스크 기반)
# =================================================================

class OrganTracker:
    """
    [수정]
    학습된 모델을 사용하여 이미지에서 장기를 탐지하고,
    바운딩 박스와 함께 '이진 마스크(binary mask)'를 반환하는 추론기
    """
    def __init__(self, model_path, device):
        self.device = device
        self.model = get_organ_detector_model() # 이 함수는 정의되어 있어야 함
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.to(device)
        self.model.eval()
        self.transform = T.Compose([T.ToTensor()])

    @torch.no_grad()
    def detect_organ(self, image_path):
        # --- [오류 처리] 손상된 이미지를 안전하게 건너뜀 ---
        try:
            image = Image.open(image_path).convert("RGB")
        except UnidentifiedImageError:
            print(f"\\n[경고] 손상된 이미지 파일(UnidentifiedImageError)을 건너뜁니다: {image_path}")
            return None, None # [수정] 튜플 반환
        except Exception as e:
            print(f"\\n[경고] 이미지 파일 처리 중 알 수 없는 오류. {image_path}, 오류: {e}")
            return None, None # [수정] 튜플 반환
        # ---------------------------------------------------
            
        image_tensor = self.transform(image).unsqueeze(0).to(self.device)
        prediction = self.model(image_tensor)
        
        scores = prediction[0]['scores']
        if len(scores) == 0:
            return None, None # [수정] 탐지된 객체 없음

        best_idx = torch.argmax(scores)
        
        # [수정] 바운딩 박스와 마스크를 모두 반환
        box = prediction[0]['boxes'][best_idx].cpu().numpy()
        
        # 마스크를 [H, W] 형태의 2D numpy 배열로 추출
        # 모델 출력 마스크는 0~1 사이의 확률 값이므로, 임계값(예: 0.5)을 적용
        mask_tensor = prediction[0]['masks'][best_idx, 0].cpu() # [H, W]
        binary_mask = (mask_tensor > 0.5).numpy() # 0.5 이상인 픽셀만 True
        
        # 마스크가 비어있는지 한 번 더 확인
        if not np.any(binary_mask):
            return box, None # 박스는 있지만 유효한 마스크가 없는 경우

        return box, binary_mask # [수정] (box, mask) 튜플 반환

def predict_camera_motion(image_dir, model_path, output_json_path):
    """
    [수정]
    이미지 시퀀스를 처리하여 장기 위치를 추적하고,
    '마스크의 무게중심(Centroid)'과 '마스크의 픽셀 면적' 변화를 기반으로
    카메라 이동을 예측하여 JSON 파일로 저장합니다.
    """
    print("\\n--- 파트 2 & 3: 카메라 이동 방향 예측 및 JSON 저장 시작 (마스크 기반) ---")
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    tracker = OrganTracker(model_path, device)
    
    image_files = sorted([p for p in Path(image_dir).glob("*.png")]) # 확장자 확인
    if len(image_files) < 2:
        print("오류: 이동을 예측하려면 2개 이상의 이미지가 필요합니다.")
        return

    # [수정] previous_box와 previous_mask를 모두 추적
    previous_box = None
    previous_mask = None 
    motion_results = []
    
    for i, img_path in tqdm(enumerate(image_files), desc="이미지 처리 및 이동 예측", total=len(image_files)):
        
        # [수정] detect_organ은 (box, mask) 튜플을 반환
        current_box, current_mask = tracker.detect_organ(img_path)
        
        dx, dy, dz = None, None, None
        
        # [수정] 마스크를 기준으로 이동 계산
        # 두 프레임 모두에서 유효한 마스크가 탐지되었는지 확인
        if current_mask is not None and previous_mask is not None and \
           np.any(current_mask) and np.any(previous_mask):
            
            # 이전 프레임 마스크의 중심(centroid)과 면적(area) 계산
            y_prev, x_prev = np.where(previous_mask) # True인 픽셀의 좌표
            prev_center_x = np.mean(x_prev)
            prev_center_y = np.mean(y_prev)
            prev_area = len(x_prev) # True 픽셀의 총 개수

            # 현재 프레임 마스크의 중심(centroid)과 면적(area) 계산
            y_curr, x_curr = np.where(current_mask) # True인 픽셀의 좌표
            curr_center_x = np.mean(x_curr)
            curr_center_y = np.mean(y_curr)
            curr_area = len(x_curr) # True 픽셀의 총 개수

            # [수정] 마스크의 중심점 이동으로 dx, dy 계산
            dx = curr_center_x - prev_center_x
            dy = curr_center_y - prev_center_y
            
            # [수정] 마스크의 실제 픽셀 면적 변화로 dz 계산
            dz = float(curr_area - prev_area)
            
        motion_results.append({
            "frame_index": i, 
            "image_id": img_path.name,
            "detected_box": current_box.tolist() if current_box is not None else None,
            # [수정] motion_from_prev의 dz 키 이름 변경 (더 명확하게)
            "motion_from_prev": {
                "dx": float(dx) if dx is not None else None,
                "dy": float(dy) if dy is not None else None,
                "dz_mask_area_change": float(dz) if dz is not None else None
            }
        })
        
        # [수정] 다음 루프를 위해 현재 상태를 저장
        previous_box = current_box
        previous_mask = current_mask

    # --- JSON 저장 ---
    final_output = {
        "images": [p.name for p in image_files],
        "meta": {
            "description": "Camera motion prediction based on organ (mask centroid and area) tracking.",
            "model_used": model_path,
            "dx_dy_unit": "pixels (centroid shift)",
            "dz_proxy_unit": "pixel_area_change (mask pixel count)"
        },
        "annotations": motion_results
    }
    
    Path(output_json_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_output, f, indent=2, ensure_ascii=False)
    
    print(f"카메라 이동 예측 결과 저장 완료: {output_json_path}")